### Cell 1 - Import dependancies

In [1]:
from pathlib import Path
import csv
import torch
from ultralytics import YOLO

### Cell 2 - Project paths

In [2]:
PROJECT = Path(r"C:\Projects\smoke_vehicle_detection")
DATA_YAML = PROJECT / "EcoWatch-Final-YOLO" / "data.yaml"

### Cell 3 - Add trained model checkpoints

In [3]:
MODELS = {
    "YOLO11n_scratch": (
        PROJECT / "runs" / "ecowatch" /
        "yolo11n_smoke_scratch" / "weights" / "best.pt"
    ),
    "YOLO11s_pretrained": (
        PROJECT / "runs" / "ecowatch" / "model_comparison" /
        "YOLO11s" / "weights" / "best.pt"
    ),
    "YOLO26s_pretrained": (
        PROJECT / "runs" / "ecowatch" / "model_comparison" /
        "YOLO26s" / "weights" / "best.pt"
    ),
}

### Cell 4 - Compare and evaluate models

In [4]:
DEVICE = 0 if torch.cuda.is_available() else "cpu"
RESULTS_DIR = PROJECT / "runs" / "ecowatch" / "final_test"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_YAML.exists():
    raise FileNotFoundError(f"Dataset YAML not found: {DATA_YAML}")

results_table = []

for model_name, weights_path in MODELS.items():
    print(f"\n{'=' * 60}")
    print(f"Evaluating: {model_name}")
    print(f"Weights: {weights_path}")
    print(f"{'=' * 60}")

    if not weights_path.exists():
        print(f"SKIPPED: Weights not found for {model_name}")
        continue

    model = YOLO(str(weights_path))

    metrics = model.val(
        data=str(DATA_YAML),
        split="test",
        imgsz=640,
        batch=8,
        device=DEVICE,
        workers=4,
        plots=True,
        project=str(RESULTS_DIR),
        name=model_name,
        exist_ok=True,
    )

    row = {
        "model": model_name,
        "precision": round(float(metrics.box.mp), 4),
        "recall": round(float(metrics.box.mr), 4),
        "mAP50": round(float(metrics.box.map50), 4),
        "mAP50-95": round(float(metrics.box.map), 4),
    }
    results_table.append(row)

    for metric, value in row.items():
        print(f"{metric}: {value}")

# Save the comparison table / ප්‍රතිඵල CSV ගොනුවකට save කිරීම
csv_path = RESULTS_DIR / "model_comparison.csv"

if results_table:
    with csv_path.open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=results_table[0].keys())
        writer.writeheader()
        writer.writerows(results_table)

    print("\nFINAL TEST COMPARISON")
    for row in results_table:
        print(row)

    print(f"\nResults saved to: {csv_path}")
else:
    print("No models were evaluated. Check the checkpoint paths.")


Evaluating: YOLO11n_scratch
Weights: C:\Projects\smoke_vehicle_detection\runs\ecowatch\yolo11n_smoke_scratch\weights\best.pt
Ultralytics 8.4.192  Python-3.13.5 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
YOLO11n summary (fused): 100 layers, 2,582,347 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access  (ping: 0.1±0.0 ms, read: 67.5±21.9 MB/s, size: 37.9 KB)
val: Scanning C:\Projects\smoke_vehicle_detection\EcoWatch-Final-YOLO\test\labels.cache... 225 images, 164 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 225/225 59.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 29/29 2.4it/s 12.2s<0.2s
                   all        225         69      0.769      0.578      0.736      0.334
Speed: 1.1ms preprocess, 6.3ms inference, 0.0ms loss, 0.6ms postprocess per image
Results saved to C:\Projects\smoke_vehicle_detection\runs\ecowatch\final_test\YOLO11n_scratch
model: YOLO11n_scratch
precision: 0

In [5]:

from pathlib import Path

ROOT = Path(r"C:\Projects\smoke_vehicle_detection\EcoWatch-Final-YOLO")
IMAGE_DIR = ROOT / "test" / "images"
LABEL_DIR = ROOT / "test" / "labels"

positive = []
negative = []

for image in IMAGE_DIR.iterdir():
    if image.suffix.lower() not in {".jpg", ".jpeg", ".png", ".webp"}:
        continue

    label = LABEL_DIR / f"{image.stem}.txt"

    if not label.exists():
        print("Missing label:", image.name)
        continue

    lines = [
        line.strip()
        for line in label.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]

    if any(line.split()[0] == "0" for line in lines):
        positive.append(image)
    else:
        negative.append(image)

print("Positive images:", len(positive))
print("Negative images:", len(negative))
print("\nSample positive images:")
for image in positive[:10]:
    print(image.name)


Positive images: 61
Negative images: 164

Sample positive images:
smoke_000109_jpg.rf.20a2c06ef55794237536ba18e1a7b472.jpg
smoke_000118_jpg.rf.437e5f1630f39c8806ba65b310e0161e.jpg
smoke_000123_jpg.rf.2c3731b85b260b6d24127c1f7f5ab5b6.jpg
smoke_000126_jpg.rf.a1144a194c84e512541551a721696639.jpg
smoke_000131_jpg.rf.5d228ed139e61e88804172083f28cb62.jpg
smoke_000157_jpg.rf.51ea124715a1d318cd03f873ab62388c.jpg
smoke_000157_jpg.rf.d4f9fcddde69aebb41849e0f9545ba2c.jpg
smoke_000167_jpg.rf.5aef4ea76846f2b9e6dc9b2040618ba3.jpg
smoke_000168_jpg.rf.189fe64c69afee777d71e0730f656459.jpg
smoke_000168_jpg.rf.1fa8040752bc97cb03e18a3bb9a9ecca.jpg


#### Test all 61 negative images

In [7]:

from pathlib import Path
import csv
import torch
from ultralytics import YOLO

PROJECT = Path(r"C:\Projects\smoke_vehicle_detection")
DATASET = PROJECT / "EcoWatch-Final-YOLO"

IMAGE_DIR = DATASET / "test" / "images"
LABEL_DIR = DATASET / "test" / "labels"

MODELS = {
    "YOLO11n_scratch": (
        PROJECT / "runs" / "ecowatch" /
        "yolo11n_smoke_scratch" / "weights" / "best.pt"
    ),
    "YOLO11s_pretrained": (
        PROJECT / "runs" / "ecowatch" / "model_comparison" /
        "YOLO11s" / "weights" / "best.pt"
    ),
    "YOLO26s_pretrained": (
        PROJECT / "runs" / "ecowatch" / "model_comparison" /
        "YOLO26s" / "weights" / "best.pt"
    ),
}

DEVICE = 0 if torch.cuda.is_available() else "cpu"
OUTPUT_DIR = PROJECT / "runs" / "ecowatch" / "positive_model_comparison"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Find test images that contain smoke labels.
positive_images = []

for image in IMAGE_DIR.iterdir():
    if image.suffix.lower() not in {".jpg", ".jpeg", ".png", ".webp"}:
        continue

    label = LABEL_DIR / f"{image.stem}.txt"

    if not label.exists():
        continue

    lines = [
        line.strip()
        for line in label.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]

    if any(line.split()[0] == "0" for line in lines):
        positive_images.append(image)

positive_images.sort()
print("Positive test images:", len(positive_images))

all_rows = []

# Evaluate every model on the same positive images.
for model_name, weights_path in MODELS.items():
    print(f"\n{'=' * 55}")
    print("Model:", model_name)

    if not weights_path.exists():
        print("Weights not found:", weights_path)
        continue

    model = YOLO(str(weights_path))
    model_output = OUTPUT_DIR / model_name
    model_output.mkdir(parents=True, exist_ok=True)

    detected_images = 0

    for image_path in positive_images:
        result = model.predict(
            source=str(image_path),
            imgsz=640,
            conf=0.25,
            device=DEVICE,
            save=True,
            project=str(model_output),
            name="predictions",
            exist_ok=True,
            verbose=False,
        )[0]

        boxes = result.boxes
        count = len(boxes)
        scores = boxes.conf.cpu().tolist() if count else []

        if count > 0:
            detected_images += 1

        all_rows.append({
            "model": model_name,
            "image": image_path.name,
            "detected_boxes": count,
            "confidence_scores": ";".join(
                f"{score:.3f}" for score in scores
            ),
        })

    print("Images with predictions:", detected_images)
    print("Images with no predictions:",
          len(positive_images) - detected_images)

csv_path = OUTPUT_DIR / "all_model_predictions.csv"

with csv_path.open("w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(
        file,
        fieldnames=[
            "model", "image", "detected_boxes", "confidence_scores"
        ],
    )
    writer.writeheader()
    writer.writerows(all_rows)

print("\nComparison CSV:", csv_path)
print("Prediction images saved under:", OUTPUT_DIR)


Positive test images: 61

Model: YOLO11n_scratch
Results saved to C:\Projects\smoke_vehicle_detection\runs\ecowatch\positive_model_comparison\YOLO11n_scratch\predictions
Results saved to C:\Projects\smoke_vehicle_detection\runs\ecowatch\positive_model_comparison\YOLO11n_scratch\predictions
Results saved to C:\Projects\smoke_vehicle_detection\runs\ecowatch\positive_model_comparison\YOLO11n_scratch\predictions
Results saved to C:\Projects\smoke_vehicle_detection\runs\ecowatch\positive_model_comparison\YOLO11n_scratch\predictions
Results saved to C:\Projects\smoke_vehicle_detection\runs\ecowatch\positive_model_comparison\YOLO11n_scratch\predictions
Results saved to C:\Projects\smoke_vehicle_detection\runs\ecowatch\positive_model_comparison\YOLO11n_scratch\predictions
Results saved to C:\Projects\smoke_vehicle_detection\runs\ecowatch\positive_model_comparison\YOLO11n_scratch\predictions
Results saved to C:\Projects\smoke_vehicle_detection\runs\ecowatch\positive_model_comparison\YOLO11n_scr

#### Test all 164 negative images

In [8]:

from pathlib import Path
import csv
import torch
from ultralytics import YOLO

PROJECT = Path(r"C:\Projects\smoke_vehicle_detection")
DATASET = PROJECT / "EcoWatch-Final-YOLO"

IMAGE_DIR = DATASET / "test" / "images"
LABEL_DIR = DATASET / "test" / "labels"

MODELS = {
    "YOLO11n_scratch": PROJECT / "runs" / "ecowatch" /
        "yolo11n_smoke_scratch" / "weights" / "best.pt",
    "YOLO11s_pretrained": PROJECT / "runs" / "ecowatch" /
        "model_comparison" / "YOLO11s" / "weights" / "best.pt",
    "YOLO26s_pretrained": PROJECT / "runs" / "ecowatch" /
        "model_comparison" / "YOLO26s" / "weights" / "best.pt",
}

DEVICE = 0 if torch.cuda.is_available() else "cpu"
OUT = PROJECT / "runs" / "ecowatch" / "negative_model_comparison"
OUT.mkdir(parents=True, exist_ok=True)

negative_images = []

for image in IMAGE_DIR.iterdir():
    if image.suffix.lower() not in {".jpg", ".jpeg", ".png", ".webp"}:
        continue

    label = LABEL_DIR / f"{image.stem}.txt"
    if not label.exists():
        continue

    lines = [
        line.strip()
        for line in label.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]

    # Empty labels mean no annotated smoke objects.
    if not any(line.split()[0] == "0" for line in lines):
        negative_images.append(image)

negative_images.sort()
print("Negative images:", len(negative_images))

summary = []
details = []

for name, weights in MODELS.items():
    if not weights.exists():
        print("Missing model:", weights)
        continue

    model = YOLO(str(weights))
    false_alarm_images = 0
    total_boxes = 0

    for img in negative_images:
        result = model.predict(
            source=str(img),
            imgsz=640,
            conf=0.25,
            device=DEVICE,
            save=True,
            project=str(OUT),
            name=name,
            exist_ok=True,
            verbose=False,
        )[0]

        count = len(result.boxes)
        total_boxes += count

        if count > 0:
            false_alarm_images += 1

        details.append({
            "model": name,
            "image": img.name,
            "predicted_boxes": count,
        })

    summary.append({
        "model": name,
        "negative_images": len(negative_images),
        "images_with_false_alarms": false_alarm_images,
        "images_without_detections": len(negative_images) - false_alarm_images,
        "total_predicted_boxes": total_boxes,
        "false_alarm_image_rate": round(
            false_alarm_images / len(negative_images), 4
        ) if negative_images else 0,
    })

    print(summary[-1])

for filename, rows in [
    ("negative_summary.csv", summary),
    ("negative_details.csv", details),
]:
    if rows:
        with (OUT / filename).open(
            "w", newline="", encoding="utf-8"
        ) as f:
            writer = csv.DictWriter(f, fieldnames=rows[0].keys())
            writer.writeheader()
            writer.writerows(rows)

print("\nSaved reports to:", OUT)


Negative images: 164
Results saved to C:\Projects\smoke_vehicle_detection\runs\ecowatch\negative_model_comparison\YOLO11n_scratch
Results saved to C:\Projects\smoke_vehicle_detection\runs\ecowatch\negative_model_comparison\YOLO11n_scratch
Results saved to C:\Projects\smoke_vehicle_detection\runs\ecowatch\negative_model_comparison\YOLO11n_scratch
Results saved to C:\Projects\smoke_vehicle_detection\runs\ecowatch\negative_model_comparison\YOLO11n_scratch
Results saved to C:\Projects\smoke_vehicle_detection\runs\ecowatch\negative_model_comparison\YOLO11n_scratch
Results saved to C:\Projects\smoke_vehicle_detection\runs\ecowatch\negative_model_comparison\YOLO11n_scratch
Results saved to C:\Projects\smoke_vehicle_detection\runs\ecowatch\negative_model_comparison\YOLO11n_scratch
Results saved to C:\Projects\smoke_vehicle_detection\runs\ecowatch\negative_model_comparison\YOLO11n_scratch
Results saved to C:\Projects\smoke_vehicle_detection\runs\ecowatch\negative_model_comparison\YOLO11n_scratc